# Part 5 · Agent Substrate: your agents in a gVisor sandbox

Parts 1-4 secured *service* and *agent* traffic. This part is about **where an agent runs**. Solo's kagent **Agent Substrate** runs each agent as an **actor inside a gVisor sandbox** on a pool of pre-warmed workers, with memory snapshots for fast resume, you deploy a `SandboxAgent` instead of an ordinary pod.

> **Beta / off by default.** Substrate ships in kagent-enterprise ≥ v0.5.2 and is disabled until you turn it on. It runs on kind with **gVisor (runsc)**, no `/dev/kvm` needed, because gVisor is a userspace kernel.

### The substrate components

`substrate-up.sh` turns on kagent's **Agent Substrate** engine (API group `ate.dev`; every component is prefixed `ate*`). What it adds to the `kagent` namespace:

| Component | Role |
|---|---|
| **SandboxAgent** (CRD) | the agent you deploy, runs as a gVisor actor, not an ordinary pod |
| **WorkerPool** (`ate.dev`) | pool of pre-warmed gVisor workers (`ateom-gvisor` image) |
| **ate-api-server** | control-plane gRPC API (`kagent-api.kagent.svc:443`), manages actors, resolves their env |
| **ate-controller** | reconciles `ActorTemplate`s / golden actors |
| **atelet** (DaemonSet) | downloads `runsc` and runs actors under gVisor on each node |
| **atenet-router** | actor networking |
| **valkey** | actor / worker state store |


## Connect · run this first

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# clear strays (dev servers, other labs' stale port-forwards) off this demo's local ports;
# Docker publishes and this suite's own kubectl forwards survive
./demo-scripts/free-ports.sh 18083
export CTX KAGENT_NS
export KENT_CRDS_CHART="oci://us-docker.pkg.dev/solo-public/kagent-enterprise-helm/charts/kagent-enterprise-crds"
export KENT_CHART="oci://us-docker.pkg.dev/solo-public/kagent-enterprise-helm/charts/kagent-enterprise"
export KAGENT_ENT_VERSION="${KAGENT_ENT_VERSION:-0.5.2}"
echo "context: $CTX ; kagent-enterprise target: $KAGENT_ENT_VERSION"
kubectl --context $CTX version -o json 2>/dev/null | python3 -c "import sys,json;print('k8s:',json.load(sys.stdin)['serverVersion']['gitVersion'],'(substrate needs >=1.33)')" 2>/dev/null || true

## How substrate gets enabled

Substrate is **off by default**. The **Enable substrate** cell below turns it on: it installs the substrate control plane and a gVisor `WorkerPool`, a pool of pre-warmed, sandboxed workers that your agents run on. It's idempotent, so it's safe to re-run. On Apple Silicon it uses the arm64 `ateom-gvisor` worker image.

In [ ]:
: "${CTX:=kind-substrate}"
# ONE-TIME: enable Agent Substrate, installs the substrate control plane + a gVisor
# WorkerPool. Idempotent (already done if you ran setup with ENABLE_SUBSTRATE=true). ~several min.
bash "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/demo-scripts/substrate-cluster.sh"

## 5.1 · Step 1: prove the agent runs in a gVisor sandbox

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 260" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="260" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Step 1 · Where your agent actually runs: a gVisor-sandboxed actor</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="14" y="66" width="116" height="46" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.5"/><text x="72" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#312e81">SandboxAgent</text><text x="72" y="101" text-anchor="middle" font-size="8" fill="#4338ca">(kagent CRD)</text><line x1="130" y1="89" x2="168" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><rect x="170" y="66" width="120" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="230" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">kagent-</text><text x="230" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">controller</text><line x1="290" y1="89" x2="330" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="310" y="80" text-anchor="middle" font-size="7.5" fill="#475569">ActorTemplate</text><rect x="332" y="66" width="110" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="387" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">ate-api-</text><text x="387" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">server</text><line x1="442" y1="89" x2="468" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="455" y="80" text-anchor="middle" font-size="7.5" fill="#475569">bind</text><rect x="470" y="52" width="236" height="120" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="588" y="70" text-anchor="middle" font-size="9" font-weight="700" fill="#14532d">WorkerPool worker · ateom-gvisor</text><rect x="486" y="84" width="204" height="74" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="588" y="102" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">gVisor sandbox (runsc)</text><text x="588" y="120" text-anchor="middle" font-size="10" font-weight="700" fill="#92400e">ADK actor</text><text x="588" y="138" text-anchor="middle" font-size="8" fill="#b45309">guest kernel ≠ host kernel</text><rect x="300" y="196" width="220" height="40" rx="8" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.4"/><text x="410" y="214" text-anchor="middle" font-size="9" font-weight="700" fill="#5b21b6">kagent-atelet (DaemonSet)</text><text x="410" y="228" text-anchor="middle" font-size="8" fill="#6d28d9">installs runsc on the node</text><line x1="520" y1="206" x2="588" y2="172" stroke="#8b5cf6" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#v)"/><text x="566" y="192" text-anchor="middle" font-size="7.5" fill="#6d28d9">runsc</text><text x="360" y="252" text-anchor="middle" font-size="10" fill="#64748b">A SandboxAgent → ActorTemplate → bound onto a pooled gVisor worker; the agent runs in a runsc sandbox with its own guest kernel.</text></svg></div>

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# substrate must be enabled first (the ‘Enable substrate’ cell above, or setup ENABLE_SUBSTRATE=true).
# deploy a SandboxAgent, it runs as a gVisor actor on the WorkerPool, not an ordinary pod.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: substrate-demo, namespace: kagent }
spec:
  type: Declarative
  description: Minimal Go ADK SandboxAgent running on Agent Substrate (gVisor).
  declarative:
    runtime: go                       # go = faster startup; python = full feature set
    modelConfig: default-model-config # auto-created from the anthropic provider
    systemMessage: "You are a helpful assistant running inside a gVisor-sandboxed actor."
  substrate:
    workerPoolRef: { name: kagent-default }
EOF
kubectl --context $CTX -n $KAGENT_NS get sandboxagent substrate-demo

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
printf '%s== the pool worker runs the gVisor ateom image ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get pods -l ate.dev/worker-pool=kagent-default -o jsonpath='{.items[*].spec.containers[*].image}'; echo
printf '%s== the WorkerPool declares sandboxClass gvisor ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get workerpool kagent-default -o jsonpath='{.spec.sandboxClass}'; echo
printf '%s== the atelet DaemonSet installed runsc on the node ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get ds kagent-atelet
kubectl --context $CTX -n $KAGENT_NS logs ds/kagent-atelet --tail=20 | grep -i runsc || true
echo
printf '%s== hardened sandbox host: the ateom-gvisor worker is distroless (no shell) ==%s\n' "$GRN$BLD" "$RST"
WPOD=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default -o name | head -1)
if kubectl --context $CTX -n $KAGENT_NS exec "${WPOD#pod/}" -- sh -c 'true' >/dev/null 2>&1; then
  echo "  a shell opened, unexpected for this image"
else
  echo "  ${GRN}✓ no shell in the worker: 'kubectl exec ... -- sh' is rejected ('sh' is not in the image)${RST}"
fi
echo "    Each actor runs as a gVisor (runsc) sandbox with its own guest kernel, isolated from the"
echo "    node and from the other actors, so there is no host shell to pivot from."

## 5.2 · Step 2: warm pool vs cold bind

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 238" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="238" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Step 2 · Warm pool vs cold bind</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="20" y="58" width="190" height="52" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/><text x="115" y="78" text-anchor="middle" font-size="9" font-weight="700" fill="#14532d">WorkerPool: 2 ready workers</text><text x="115" y="94" text-anchor="middle" font-size="8" fill="#166534">warm, golden resident</text><line x1="210" y1="84" x2="300" y2="84" stroke="#16a34a" stroke-width="1.7" marker-end="url(#g)"/><text x="255" y="76" text-anchor="middle" font-size="7.5" fill="#166534">chat</text><rect x="302" y="64" width="150" height="40" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.4"/><text x="377" y="80" text-anchor="middle" font-size="8.5" font-weight="700" fill="#14532d">bind to warm worker</text><text x="377" y="94" text-anchor="middle" font-size="7.5" fill="#166534">resume from golden</text><rect x="470" y="64" width="90" height="40" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="515" y="88" text-anchor="middle" font-size="13" font-weight="700" fill="#14532d">~1s</text><rect x="20" y="150" width="190" height="52" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/><text x="115" y="170" text-anchor="middle" font-size="9" font-weight="700" fill="#334155">WorkerPool: 0 (scaled down)</text><text x="115" y="186" text-anchor="middle" font-size="8" fill="#475569">cold</text><line x1="210" y1="176" x2="300" y2="176" stroke="#d97706" stroke-width="1.7" marker-end="url(#d)"/><text x="255" y="168" text-anchor="middle" font-size="7.5" fill="#92400e">chat</text><rect x="302" y="156" width="150" height="40" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="377" y="171" text-anchor="middle" font-size="7.5" font-weight="700" fill="#7c2d12">schedule pod → ateom</text><text x="377" y="184" text-anchor="middle" font-size="7.5" fill="#92400e">start → runsc spawn → bind</text><rect x="470" y="156" width="140" height="40" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="540" y="180" text-anchor="middle" font-size="13" font-weight="700" fill="#7c2d12">~10–40s</text><text x="360" y="224" text-anchor="middle" font-size="10" fill="#64748b">Two live timings: a warm worker resumes fast; a cold pool must schedule a pod, start ateom and spawn the runsc actor. Numbers vary on a laptop.</text></svg></div>

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# WARM: the pool already holds pre-warmed workers, a new actor binds onto one, no pod to schedule.
kubectl --context $CTX -n $KAGENT_NS wait pod -l ate.dev/worker-pool=kagent-default --for=condition=Ready --timeout=120s
echo "warm workers ready: $(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default --no-headers | grep -c Running)"
echo "a bind onto a warm worker resumes from the golden snapshot, sub-second (5.3 proves it)."

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# COLD: scale the pool to 0, then bringing a worker back forces pod schedule + ateom start +
# runsc actor spawn. Wait on the worker pods being DELETED, not on status.replicas=0 , 
# 'wait --for=jsonpath {.status.replicas}=0' races the status update and hangs.
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":0}}'
kubectl --context $CTX -n $KAGENT_NS wait pod -l ate.dev/worker-pool=kagent-default --for=delete --timeout=120s
echo "pool cold (0 workers). Now bring one back and time the cold path:"
time ( kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":1}}' >/dev/null
       until kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default -o name 2>/dev/null | grep -q pod/; do sleep 1; done
       kubectl --context $CTX -n $KAGENT_NS wait pod -l ate.dev/worker-pool=kagent-default --for=condition=Ready --timeout=300s )
echo "cold path = pod schedule + ateom start + runsc actor spawn + bind. Restore 2 replicas:"
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":2}}'


## 5.3 · Actors are cheap, pods are not

Two properties matter here: the workload really is **sandboxed**, and standing up another agent does **not** cost another pod. Deploy a few more SandboxAgents against the same WorkerPool and each reaches Ready in about a second while the worker-pod count stays flat. The gVisor actors pack onto the workers already running, each resuming from its golden snapshot. Many isolated actors, few pods.

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
before=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default --no-headers | grep -c Running)
printf '%s== worker pods before: %s ==%s\n' "$CYN$BLD" "$before" "$RST"
printf '%s== spin up 3 more sandboxed actors, timing each bind ==%s\n' "$CYN$BLD" "$RST"
for n in 2 3 4; do
  t0=$SECONDS
  kubectl --context $CTX apply -f - >/dev/null <<EOF
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: substrate-demo-$n, namespace: kagent }
spec:
  type: Declarative
  description: density demo actor $n
  declarative: { runtime: go, modelConfig: default-model-config, systemMessage: "actor $n" }
  substrate: { workerPoolRef: { name: kagent-default } }
EOF
  kubectl --context $CTX -n $KAGENT_NS wait sandboxagent/substrate-demo-$n --for=condition=Ready --timeout=90s >/dev/null
  printf '   substrate-demo-%s Ready in %ss\n' "$n" "$((SECONDS-t0))"
done
echo
printf '%s== the payoff: N gVisor actors, same worker pods ==%s\n' "$GRN$BLD" "$RST"
echo "gVisor golden actors (ActorTemplates, CLASS gvisor):"
kubectl --context $CTX -n $KAGENT_NS get actortemplates
after=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default --no-headers | grep -c Running)
printf '%sworker pods after: %s%s  (4 sandboxed actors packed onto %s pods, each bound in ~1s)\n' "$GRN$BLD" "$after" "$RST" "$after"
echo
echo "clean up the extras (leave substrate-demo):"
kubectl --context $CTX -n $KAGENT_NS delete sandboxagent substrate-demo-2 substrate-demo-3 substrate-demo-4

## 5.4 · golden actor + snapshot resume

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 232" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="232" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Golden actor + snapshot resume</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="280" y="64" width="160" height="54" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="360" y="86" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">golden actor</text><text x="360" y="102" text-anchor="middle" font-size="8" fill="#166534">memory snapshot on pause</text><rect x="500" y="66" width="200" height="50" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="600" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#334155">gs:// snapshot bucket</text><text x="600" y="101" text-anchor="middle" font-size="8" fill="#475569">cross-worker persistence</text><line x1="440" y1="86" x2="498" y2="86" stroke="#334155" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#n)"/><text x="469" y="78" text-anchor="middle" font-size="7.5" fill="#475569">snapshot</text><rect x="160" y="150" width="240" height="50" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/><text x="280" y="170" text-anchor="middle" font-size="10" font-weight="700" fill="#1e293b">new actor bind</text><text x="280" y="186" text-anchor="middle" font-size="8.5" fill="#475569">ResumeGoldenActor</text><line x1="360" y1="118" x2="300" y2="148" stroke="#16a34a" stroke-width="1.7" marker-end="url(#g)"/><text x="345" y="138" text-anchor="middle" font-size="7.5" fill="#166534">resume</text><rect x="430" y="150" width="270" height="50" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="565" y="169" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">internal engine reconcile</text><text x="565" y="185" text-anchor="middle" font-size="8" fill="#92400e">no operator verb · automatic</text><text x="360" y="222" text-anchor="middle" font-size="10" fill="#64748b">Idle actor snapshots its memory, then resumes from a golden snapshot, not a cold boot. Automatic, no kubectl verb.</text></svg></div>

Pause, snapshot and resume are **automatic**: there is no kubectl verb, `ResumeGoldenActor` is an internal phase of the substrate engine. This is what you just saw. A new actor resumes from a **golden memory snapshot** instead of a cold boot, which is why the binds in 5.2 and 5.3 were sub-second.

Resuming from a golden works with no configuration. To persist snapshots **across workers**, so an actor can resume on a different worker or after a restart, point the pool at object storage:

```yaml
spec:
  substrate:
    workerPoolRef: { name: kagent-default }
    snapshotsConfig:
      location: gs://<your-bucket>/kagent/substrate-demo/
```

## 5.5 · The worker fleet

A `WorkerPool` is a fleet of pre-warmed worker pods, and every agent runs as a gVisor actor packed onto one of them. Adding an agent binds a new actor onto a warm worker in about a second; it does not cost a pod. The fleet scales on its own, so you add workers for capacity without touching the agents.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="760" height="300" rx="10" fill="#f8fafc"/><text x="380" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Agent Substrate: one WorkerPool, many gVisor actors</text><text x="380" y="44" text-anchor="middle" font-size="10.5" fill="#475569">each SandboxAgent runs as a gVisor actor; actors pack onto shared, pre-warmed worker pods</text><rect x="20" y="58" width="720" height="192" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="1.6"/><text x="36" y="78" font-size="11" font-weight="700" fill="#1e40af">WorkerPool · kagent-default · sandboxClass gvisor</text><rect x="36" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="144" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 1</text><rect x="54" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="144" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · checkout</text><text x="144" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="54" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="144" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · search</text><text x="144" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="272" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="380" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 2</text><rect x="290" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="380" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · pricing</text><text x="380" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="290" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="380" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · support</text><text x="380" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="508" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="616" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 3</text><rect x="526" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="616" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · summarizer</text><text x="616" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="526" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="616" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · planner</text><text x="616" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><text x="380" y="278" text-anchor="middle" font-size="10.5" fill="#334155">Add an agent and a new actor binds onto a warm worker in about a second, not a new pod. Scale the pool to add workers.</text></svg></div>

Below: the live fleet and an elastic resize (2 to 4 workers and back, with the actors untouched).

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
printf '%s== the worker fleet, each pod hosts gVisor actors ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get pods -l ate.dev/worker-pool=kagent-default -o wide

printf '%s== the fleet is elastic: scale 2 -> 4 workers, actors untouched ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":4}}' >/dev/null
n=0
for i in $(seq 1 30); do
  n=$(kubectl --context $CTX -n $KAGENT_NS get pods -l ate.dev/worker-pool=kagent-default --field-selector=status.phase=Running --no-headers 2>/dev/null | wc -l | tr -d ' ')
  [ "$n" -ge 4 ] && break
  sleep 2
done
actors=$(kubectl --context $CTX -n $KAGENT_NS get actortemplates --no-headers | wc -l | tr -d ' ')
echo "  fleet grew to $n worker pods; gVisor actors still: $actors (unaffected)"
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":2}}' >/dev/null
echo "  scaled back to 2, workers are cattle, the actors carried on"

## 5.6 · Chat with the sandboxed agent

The proof that matters: talk to it. A SandboxAgent is an A2A server. Regular agents answer on `/api/a2a/<ns>/<name>/`; sandboxed ones answer on `/api/a2a-sandboxes/<ns>/<name>/`. Every message carries a `contextId`, which is a kagent session id, so we open a session and then send the prompt. The reply comes straight from the ADK agent running inside the gVisor actor.

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
kubectl --context $CTX -n $KAGENT_NS port-forward svc/kagent-controller 18083:8083 >/dev/null 2>&1 &
PF=$!
for i in $(seq 1 10); do curl -s -o /dev/null http://localhost:18083/api/sessions 2>/dev/null && break; sleep 1; done

ask() {
  # 1) open a session; its id is the A2A contextId a sandbox actor requires
  local sid=$(curl -s --max-time 15 -X POST http://localhost:18083/api/sessions \
      -H 'content-type: application/json' -d '{"agent_ref":"kagent/substrate-demo","name":"chat"}' \
      | python3 -c 'import sys,json; print(json.load(sys.stdin)["data"]["id"])')
  # 2) send the prompt over A2A (message/send) to the gVisor-sandboxed actor
  local req=$(python3 -c "import json,sys; print(json.dumps({'jsonrpc':'2.0','id':'1','method':'message/send','params':{'message':{'role':'user','parts':[{'kind':'text','text':sys.argv[1]}],'messageId':'m1','contextId':sys.argv[2]}}}))" "$1" "$sid")
  printf '%s> %s%s\n' "$CYN$BLD" "$1" "$RST"
  curl -s --max-time 90 -X POST http://localhost:18083/api/a2a-sandboxes/kagent/substrate-demo/ \
      -H 'content-type: application/json' -d "$req" \
    | python3 -c '
import sys,json
d=json.load(sys.stdin); r=d.get("result",{})
arts=r.get("artifacts",[])
if arts:
    print("  "+arts[0]["parts"][0]["text"])
    app=next((m.get("metadata",{}).get("adk_app_name") for m in r.get("history",[]) if m.get("metadata",{}).get("adk_app_name")),None)
    if app: print("     (answered by "+app+", running as a gVisor actor)")
else:
    print("  (no answer: "+str(d.get("error",{}).get("message","?"))+")")'
}

ask "In one sentence, what is 17 times 3?"
ask "Name one benefit of running an agent in a gVisor sandbox. One sentence."

kill $PF 2>/dev/null; wait $PF 2>/dev/null || true

## Tear down

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
kubectl --context $CTX -n $KAGENT_NS delete sandboxagent substrate-demo --ignore-not-found
# to fully remove substrate (and hand kagent back to Part 4):
#   helm --kube-context $CTX upgrade kagent "$KENT_CHART" -n $KAGENT_NS --reuse-values \
#     --set substrate.enabled=false --set substrateWorkerPool.create=false --set controller.substrate.enabled=false
#   kubectl --context $CTX -n $KAGENT_NS delete workerpool kagent-default --ignore-not-found